<a href="https://colab.research.google.com/github/Onureeva/airbnb-vienna-nlp-pricing/blob/main/Reviews_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review Cleaning and Analysis

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")



In [ ]:
# import Airbnb Listing datasets for Asheville

listings_url = 'https://data.insideairbnb.com/austria/vienna/vienna/2025-09-14/data/reviews.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')



In [ ]:
# Ex1 a: Display the first 10 rows

listings_df.head(10)

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,40625,73717,2010-08-04,176849,William,Ingela is a superb host. She personally welco...
1,40625,110809,2010-10-03,222519,Kerri,Ingela was a perfect host! She gave great dire...
2,40625,206046,2011-03-22,273895,Heather,Our stay in Vienna with Ingela could not have ...
3,40625,554329,2011-09-21,254998,Fernando,Our stay in the beautiful city of Vienna was g...
4,40625,584745,2011-10-01,314952,Michael,We really enjoyed our visit and loved the very...
5,40625,756907,2011-12-01,148764,Julie,"My experience was great, with a really lovely ..."
6,40625,782765,2011-12-13,1386050,Christian,I totally agree with the reviews of the previo...
7,40625,851835,2012-01-09,1230408,Kristina & Oleg,We spent several days in Vienna in January 201...
8,40625,2510231,2012-10-05,2616415,Sue,"This was a very pleasant flat, just as describ..."
9,40625,2648798,2012-10-18,3444804,Pui Yee,Ingela was a great host. Though we did not mee...


In [ ]:
listings_df.shape

(612430, 6)

In [ ]:
# Ex1 b: Explore columns, data types, and non-null counts

!pip install langdetect
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
listings_df.info()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=5fb96d0caeacd0019a92a120ea2471ffa1c03bdf93f4d523c2e984079e7f1483
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612430 entries, 0 to 612429
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   listing_id     612430 non-null  int64 
 1   id             612430 non-null  int64 
 2   date           612430 non-null  object
 3   reviewer_id    612430 non-null  int64 
 4   reviewer_name  612427 non-null  object
 5   comments       612395 non-null  object
dtypes: int64(3), object(3)
memory usage: 28.0+ MB


In [ ]:
sample_df=listings_df.sample(50000, random_state=42)

In [ ]:
from numpy.random.mtrand import sample
def detect_language(text):
    try:
        # Ensure the input is treated as a string, handling non-string types (like NaN)
        if pd.isna(text): # Check for NaN values
            return None
        return detect(str(text)) # Convert to string before detection
    except LangDetectException:
        return None

listings_df['language']=listings_df['comments'].apply(detect_language)
listings_df['language'].value_counts()


,count
language,
en,326168
de,144949
fr,33382
es,21632
it,15566
ko,8281
nl,6767
ru,6449
tr,4549


In [ ]:
listings_df['language'].nunique()

45

2. DATASET CLEANING

In [ ]:
listings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612430 entries, 0 to 612429
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   listing_id     612430 non-null  int64 
 1   id             612430 non-null  int64 
 2   date           612430 non-null  object
 3   reviewer_id    612430 non-null  int64 
 4   reviewer_name  612427 non-null  object
 5   comments       612395 non-null  object
 6   language       609059 non-null  object
dtypes: int64(3), object(4)
memory usage: 32.7+ MB


In [ ]:
#Drop columns id, reviewer_id, reviewer_name
df=listings_df.drop(['id', 'reviewer_id', 'reviewer_name'], axis=1)


In [ ]:
#Choose only english comments
english_df=df[df['language']=='en']

In [ ]:
english_df.shape

(326168, 4)

In [ ]:
#Drop rows with missing values as test is essential for analysis
df=english_df.dropna(subset=['comments'])
dr=df[df['comments'].str.strip() != '']


In [ ]:
#delete duplicates
df=df.drop_duplicates(subset=['listing_id', 'comments', 'date'])

In [ ]:
#Change data format
df['date'] = pd.to_datetime(df['date'])

In [ ]:
import re

def clean_text(text):
  text = text.lower()
  text=re.sub(r'[^\w\s]', '', text) #remove punctuation
  text = re.sub(r'\d+', '', text) #remove digits
  text= text.strip() #remove leading and trailing spaces
  return text

df['comments'] = df['comments'].apply(clean_text)
#

In [ ]:
#Filter only long reviews
df['review_length'] = df['comments'].apply(lambda x: len(x.split()))
df = df[df['review_length']>2]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 319598 entries, 0 to 612429
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   listing_id     319598 non-null  int64         
 1   date           319598 non-null  datetime64[ns]
 2   comments       319598 non-null  object        
 3   language       319598 non-null  object        
 4   review_length  319598 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 14.6+ MB


In [ ]:
df.isnull().sum()

,0
listing_id,0
date,0
comments,0
language,0
review_length,0


In [ ]:
#saving cleaned dataset
df.to_csv('cleaned_reviews.csv', index=False)

In [ ]:
#To save this file on your computer, use the code:
# from google.colab import files
# files.download('cleaned_reviews.csv')

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')